In [ ]:
import sys
import os
import re
import time
import subprocess
import itertools
import tempfile
import random
import numpy as np
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import cross_val_score

BASE_DIR = os.getcwd()
SET_ENV_PATH = os.path.join("~", ".set_env.sh")
INPUT_DIRECTORY = os.path.join(BASE_DIR, "images", "original")
PROFILE_LOG_DIR = os.path.join(BASE_DIR, "logs", "profile")
os.makedirs(PROFILE_LOG_DIR, exist_ok=True)

print(sys.executable)

print(INPUT_DIRECTORY)
print(os.listdir(INPUT_DIRECTORY))

print(SET_ENV_PATH)

/users/eleves-b/2025/ivan.banny/miniconda3/bin/python
/users/eleves-b/2025/ivan.banny/code/sobelf/images/original
['australian-flag-large.gif', '051009.vince.gif', '1.gif', '200_s.gif', '9815573.gif', 'Campusplan-Hausnr.gif', 'Mandelbrot-large.gif', 'Campusplan-Mobilitaetsbeschraenkte.gif', 'Produits_sous_linux.gif', 'TimelyHugeGnu.gif', 'fire.gif', 'frame_002.gif', 'giphy-3.gif', 'porsche.gif', 'walla.gif', 'tumblr_myxlbtwVEb1qzt4vjo1_r14_500_large.gif', 'wallace.gif', 'walle.gif']
~/.set_env.sh


In [2]:
def submit_job(cmd, log_file, nodes=1, ntasks=1, cpus_per_task=1, time="00:10:00"):
    """Submit a job via sbatch, return job ID."""
    sbatch_cmd = (
        f"source {SET_ENV_PATH} && "
        f"sbatch --parsable -N {nodes} -n {ntasks} -c {cpus_per_task} "
        f"-t {time} -o {log_file} --wrap '{cmd}'"
    )
    result = subprocess.run(
        sbatch_cmd, 
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        shell=True, 
        executable="/bin/bash"
    )
    output = result.stdout.decode().strip()
    try:
        return int(output.strip().split("\n")[-1])
    except ValueError:
        print(f"sbatch failed: {output}")
        return None

def parse_sobel_filter_output(output):
    match = re.search(r"SOBEL done in ([0-9.]+) s", output)
    if match:
        return float(match.group(1))
    else:
        raise ValueError("couldn't parse output :\n" + output + '\n')

def get_gif_info(path, input_dir=INPUT_DIRECTORY):
    sizes = []
    with Image.open(os.path.join(input_dir, path)) as im:
        num_frames = im.n_frames
        for frame_index in range(num_frames):
            im.seek(frame_index)
            sizes.append(im.size)
    return num_frames, sizes

def make_log_path(impl, gif_file, run=0, **params):
    tag = '_'.join(f"{k}{v}" for k, v in params.items())
    name = gif_file.replace(".gif", "")
    return os.path.join(PROFILE_LOG_DIR, f"{impl}_{name}_{tag}_run{run}.log")

In [3]:
def submit_all_jobs(gif_files, configs, input_dir, n_runs):
    """Submit all jobs, return list of job descriptors."""
    jobs = []

    for gif_file in gif_files:
        input_path = os.path.join(input_dir, gif_file)

        for run in range(n_runs):
            seen = set()
            for rank, thread, node, cuda in configs:
                # --- SEQ ---
                key = ("SEQ", gif_file)
                if key not in seen:
                    seen.add(key)
                    log = make_log_path("seq", gif_file, run=run)
                    cmd = f"source {SET_ENV_PATH} && ./sobelf_seq {input_path} /dev/null"
                    job_id = submit_job(cmd, log, ntasks=1)
                    if job_id is not None:
                        jobs.append({"job_id": job_id, "log": log, "impl": "SEQ",
                                     "gif": gif_file, "run": run})

                # --- MPI ---
                key = ("MPI", gif_file, rank, node)
                if key not in seen:
                    seen.add(key)
                    log = make_log_path("mpi", gif_file, run=run, r=rank, n=node)
                    cmd = f"source {SET_ENV_PATH} && mpirun ./sobelf_mpi {input_path} /dev/null"
                    job_id = submit_job(cmd, log, nodes=node, ntasks=rank)
                    if job_id is not None:
                        jobs.append({"job_id": job_id, "log": log, "impl": "MPI",
                                     "gif": gif_file, "rank": rank, "node": node, "run": run})

                # --- OMP ---
                key = ("OMP", gif_file, thread)
                if key not in seen:
                    seen.add(key)
                    log = make_log_path("omp", gif_file, run=run, t=thread)
                    cmd = (f"source {SET_ENV_PATH} && OMP_NUM_THREADS={thread} "
                           f"./sobelf_omp {input_path} /dev/null")
                    job_id = submit_job(cmd, log, ntasks=1, cpus_per_task=thread)
                    if job_id is not None:
                        jobs.append({"job_id": job_id, "log": log, "impl": "OMP",
                                     "gif": gif_file, "thread": thread, "run": run})

                # --- HYB ---
                key = ("HYB", gif_file, rank, thread, node)
                if key not in seen:
                    seen.add(key)
                    log = make_log_path("hyb", gif_file, run=run, r=rank, t=thread, n=node)
                    cmd = (f"source {SET_ENV_PATH} && OMP_NUM_THREADS={thread} "
                           f"mpirun ./sobelf_hyb {input_path} /dev/null")
                    job_id = submit_job(cmd, log, nodes=node, ntasks=rank, cpus_per_task=thread)
                    if job_id is not None:
                        jobs.append({"job_id": job_id, "log": log, "impl": "HYB",
                                     "gif": gif_file, "rank": rank, "thread": thread, "node": node, "run": run})

                # --- CUDA ---
                key = ("CUDA", gif_file)
                if key not in seen:
                    seen.add(key)
                    log = make_log_path("cuda", gif_file, run=run)
                    cmd = f"source {SET_ENV_PATH} && ./sobelf_cuda {input_path} /dev/null"
                    job_id = submit_job(cmd, log, ntasks=1)
                    if job_id is not None:
                        jobs.append({"job_id": job_id, "log": log, "impl": "CUDA",
                                     "gif": gif_file, "run": run})

    print(f"Submitted {len(jobs)} jobs")
    return jobs

def wait_for_jobs(jobs, ping=8):
    """Ping until all jobs are done."""
    job_ids = {j["job_id"] for j in jobs}
    while True:
        result = subprocess.run(
            f"squeue -j {','.join(str(j) for j in job_ids)} -h -t PD,R | wc -l",
            stdout=subprocess.PIPE, shell=True, executable="/bin/bash"
        )
        rem = int(result.stdout.decode().strip())
        print(f"\r{rem}/{len(job_ids)} jobs still running...", end="", flush=True)
        if rem == 0:
            break
        time.sleep(ping)
    print("\nAll jobs done")

In [4]:
def collect_results(jobs):
    """Parse logs, build a lookup dict with median over runs."""
    raw = {}  # key -> list of times
    errors = []
    for j in jobs:
        try:
            with open(j["log"]) as f:
                text = f.read()
            t = parse_sobel_filter_output(text)

            key = (j["impl"], j["gif"])
            if j["impl"] == "MPI":
                key += (j["rank"], j["node"])
            elif j["impl"] == "OMP":
                key += (j["thread"],)
            elif j["impl"] == "HYB":
                key += (j["rank"], j["thread"], j["node"])
            # SEQ and CUDA have no extra params

            raw.setdefault(key, []).append(t)
        except Exception as e:
            errors.append({**j, "error": str(e)})

    # Take median across runs
    results = {k: np.median(v) for k, v in raw.items()}
    return results, errors

feature_names = ["num_frames", "width", "height", "total_pixels",
                 "pixels_per_frame", "ranks", "threads", "nodes",
                 "cuda_available"]
labels = {"MPI": 0, "OMP": 1, "HYB": 2, "CUDA": 3, "SEQ": 4}

# Preference order when implementations are within margin:
# simpler is better (less overhead to set up / fewer resources wasted)
SIMPLICITY_ORDER = ["SEQ", "OMP", "CUDA", "MPI", "HYB"]

def create_dataset_from_results(results, gif_files, configs, input_dir,
                                margin=0.10):
    """Build (X, y) dataset. When the best and runner-up are within `margin`
    relative difference, prefer the simpler implementation."""
    X, y = [], []
    for gif_file in gif_files:
        num_frames, sizes = get_gif_info(gif_file, input_dir=input_dir)
        width, height = sizes[0]
        total_pixels = width * height * num_frames
        pixels_per_frame = width * height

        for rank, thread, node, cuda in configs:
            times = {
                "SEQ": results.get(("SEQ", gif_file), np.inf),
                "MPI": results.get(("MPI", gif_file, rank, node), np.inf),
                "OMP": results.get(("OMP", gif_file, thread), np.inf),
                "HYB": results.get(("HYB", gif_file, rank, thread, node), np.inf),
                "CUDA": results.get(("CUDA", gif_file), np.inf) if cuda else np.inf,
            }
            best_time = min(times.values())
            if best_time == np.inf:
                continue
            # All implementations within margin of the best
            within_margin = [
                impl for impl, t in times.items()
                if t <= best_time * (1.0 + margin)
            ]
            # Pick simplest among those within margin
            best = min(within_margin, key=lambda impl: SIMPLICITY_ORDER.index(impl))

            X.append([num_frames, width, height, total_pixels,
                      pixels_per_frame, rank, thread, node, int(cuda)])
            y.append(labels[best])
    return np.array(X), np.array(y)

In [5]:
# Compile with export skip
result = subprocess.run(
    f"source {SET_ENV_PATH} && make clean && make SKIP_EXPORT=1",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    shell=True,
    executable="/bin/bash"
)
print(result.stdout)

b'rm -f sobelf_seq sobelf_mpi sobelf_omp sobelf_hyb sobelf_cuda obj/*.o\nmpicc -O3 -fopenmp -Iinclude -c -o obj/dgif_lib.o src/dgif_lib.c\nmpicc -O3 -fopenmp -Iinclude -c -o obj/egif_lib.o src/egif_lib.c\nmpicc -O3 -fopenmp -Iinclude -c -o obj/gif_err.o src/gif_err.c\nmpicc -O3 -fopenmp -Iinclude -c -o obj/gif_font.o src/gif_font.c\nmpicc -O3 -fopenmp -Iinclude -c -o obj/gif_hash.o src/gif_hash.c\nmpicc -O3 -fopenmp -Iinclude -c -o obj/gifalloc.o src/gifalloc.c\nmpicc -O3 -fopenmp -Iinclude -c -o obj/openbsd-reallocarray.o src/openbsd-reallocarray.c\nmpicc -O3 -fopenmp -Iinclude -c -o obj/quantize.o src/quantize.c\nmpicc -O3 -fopenmp -Iinclude -Wall -Wextra -DSKIP_EXPORT -c -o obj/gif_io.o src/gif_io.c\nmpicc -O3 -fopenmp -Iinclude -Wall -Wextra -DSKIP_EXPORT -c -o obj/main_seq.o src/main_seq.c\nmpicc -O3 -fopenmp -Iinclude -Wall -Wextra -DSKIP_EXPORT -o sobelf_seq obj/dgif_lib.o obj/egif_lib.o obj/gif_err.o obj/gif_font.o obj/gif_hash.o obj/gifalloc.o obj/openbsd-reallocarray.o obj/qu

In [6]:
ranks = [1, 2, 4, 8, 16]
threads = [1, 2, 4, 8, 16]
nodes = [1, 2, 4]
cudas = [True, False]
n_runs = 5
margin = 0.10
max_depth = 5
min_samples_leaf = 5
cv_folds = 5

gif_files = [f for f in os.listdir(INPUT_DIRECTORY) if f.lower().endswith(".gif")]

# Use full config product (no sampling)
configs = list(itertools.product(ranks, threads, nodes, cudas))
print(f"{len(configs)} configs x {len(gif_files)} GIFs x {n_runs} runs")

# Clean up
result = subprocess.run(
    ["./clean_test.sh"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(result.stdout)

# Submit
jobs = submit_all_jobs(gif_files, configs, INPUT_DIRECTORY, n_runs)

# Wait
wait_for_jobs(jobs)

# Collect
results, errors = collect_results(jobs)
print(f"{len(errors)} errors")

# Summarize errors by type
from collections import Counter
error_keys = Counter()
for e in errors:
    key = (e["impl"], e.get("node", 1))
    error_keys[key] += 1
print("Errors by (impl, nodes):", dict(error_keys))

150 configs x 18 GIFs x 5 runs
b'Cleaning logs/*\nCleaning images/processed/*\n'
sbatch failed: sbatch: error: Batch job submission failed: Requested node configuration is not available
sbatch failed: sbatch: error: Batch job submission failed: Requested node configuration is not available
sbatch failed: sbatch: error: Batch job submission failed: Requested node configuration is not available
sbatch failed: sbatch: error: Batch job submission failed: Requested node configuration is not available
sbatch failed: sbatch: error: Batch job submission failed: Requested node configuration is not available
sbatch failed: sbatch: error: Batch job submission failed: Requested node configuration is not available
sbatch failed: sbatch: error: Batch job submission failed: Requested node configuration is not available
sbatch failed: sbatch: error: Batch job submission failed: Requested node configuration is not available
sbatch failed: sbatch: error: Batch job submission failed: Requested node confi

In [7]:
X, y = create_dataset_from_results(results, gif_files, configs, INPUT_DIRECTORY,
                                   margin=margin)
print(f"Dataset: {len(X)} samples, label distribution: {dict(Counter(y))}")

# Find optimal ccp_alpha via cross-validation
base_tree = DecisionTreeClassifier(max_depth=max_depth,
                                   min_samples_leaf=min_samples_leaf)
path = base_tree.cost_complexity_pruning_path(X, y)
ccp_alphas = path.ccp_alphas

best_alpha, best_score = 0, 0
for alpha in ccp_alphas:
    t = DecisionTreeClassifier(max_depth=max_depth,
                               min_samples_leaf=min_samples_leaf,
                               ccp_alpha=alpha)
    scores = cross_val_score(t, X, y, cv=cv_folds)
    if scores.mean() > best_score:
        best_score = scores.mean()
        best_alpha = alpha
print(f"Best ccp_alpha: {best_alpha:.6f}, CV accuracy: {best_score:.3f}")

# Train final tree with best alpha
tree = DecisionTreeClassifier(max_depth=max_depth,
                              min_samples_leaf=min_samples_leaf,
                              ccp_alpha=best_alpha)
tree.fit(X, y)

# Final CV score
scores = cross_val_score(tree, X, y, cv=cv_folds)
print(f"Final CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")

# Print tree
class_names = [k for k, v in sorted(labels.items(), key=lambda x: x[1])]
tree_text = export_text(tree, feature_names=feature_names, class_names=class_names)
print("Decision tree :")
print(tree_text)

Dataset: 2700 samples, label distribution: {np.int64(3): 739, np.int64(0): 417, np.int64(1): 1223, np.int64(2): 70, np.int64(4): 251}
Best ccp_alpha: 0.004854, CV accuracy: 0.661
Final CV accuracy: 0.661 +/- 0.117
Decision tree :
|--- cuda_available <= 0.50
|   |--- threads <= 3.00
|   |   |--- width <= 115.00
|   |   |   |--- class: SEQ
|   |   |--- width >  115.00
|   |   |   |--- nodes <= 1.50
|   |   |   |   |--- class: MPI
|   |   |   |--- nodes >  1.50
|   |   |   |   |--- ranks <= 3.00
|   |   |   |   |   |--- class: MPI
|   |   |   |   |--- ranks >  3.00
|   |   |   |   |   |--- class: OMP
|   |--- threads >  3.00
|   |   |--- total_pixels <= 61700.00
|   |   |   |--- width <= 115.00
|   |   |   |   |--- class: SEQ
|   |   |   |--- width >  115.00
|   |   |   |   |--- threads <= 6.00
|   |   |   |   |   |--- class: OMP
|   |   |   |   |--- threads >  6.00
|   |   |   |   |   |--- class: SEQ
|   |   |--- total_pixels >  61700.00
|   |   |   |--- class: OMP
|--- cuda_available > 

In [8]:
def export_tree_as_c(tree, feature_names, class_names, func_name="select_impl"):
    """Export a fitted DecisionTreeClassifier as a C function."""
    from sklearn.tree import _tree
    tree_ = tree.tree_
    lines = []
    lines.append(f'const char* {func_name}(int {", int ".join(feature_names)}) {{')

    def recurse(node, indent):
        pad = "    " * indent
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name = feature_names[tree_.feature[node]]
            threshold = tree_.threshold[node]
            lines.append(f'{pad}if ({name} <= {threshold:.1f}) {{')
            recurse(tree_.children_left[node], indent + 1)
            lines.append(f'{pad}}} else {{')
            recurse(tree_.children_right[node], indent + 1)
            lines.append(f'{pad}}}')
        else:
            class_idx = tree_.value[node].argmax()
            lines.append(f'{pad}return "{class_names[class_idx]}";')

    recurse(0, 1)
    lines.append("}")
    return "\n".join(lines)

c_code = export_tree_as_c(tree, feature_names, class_names)
print(c_code)

with open("decision_tree.c", "w") as f:
    f.write(c_code + "\n")
print("\nSaved to decision_tree.c")

const char* select_impl(int num_frames, int width, int height, int total_pixels, int pixels_per_frame, int ranks, int threads, int nodes, int cuda_available) {
    if (cuda_available <= 0.5) {
        if (threads <= 3.0) {
            if (width <= 115.0) {
                return "SEQ";
            } else {
                if (nodes <= 1.5) {
                    return "MPI";
                } else {
                    if (ranks <= 3.0) {
                        return "MPI";
                    } else {
                        return "OMP";
                    }
                }
            }
        } else {
            if (total_pixels <= 61700.0) {
                if (width <= 115.0) {
                    return "SEQ";
                } else {
                    if (threads <= 6.0) {
                        return "OMP";
                    } else {
                        return "SEQ";
                    }
                }
            } else {
                return "OMP";
   

In [12]:
# Regret analysis
# For each sample, compare the time of the tree's pick vs the actual best.
imp = {v: k for k, v in labels.items()}
regret_records = []  # (regret, pred_impl, best_impl, best_time, pred_time, gif, config)

for gif_file in gif_files:
    num_frames, sizes = get_gif_info(gif_file, input_dir=INPUT_DIRECTORY)
    width, height = sizes[0]
    total_pixels = width * height * num_frames
    pixels_per_frame = width * height

    for rank, thread, node, cuda in configs:
        times = {
            "SEQ": results.get(("SEQ", gif_file), np.inf),
            "MPI": results.get(("MPI", gif_file, rank, node), np.inf),
            "OMP": results.get(("OMP", gif_file, thread), np.inf),
            "HYB": results.get(("HYB", gif_file, rank, thread, node), np.inf),
            "CUDA": results.get(("CUDA", gif_file), np.inf) if cuda else np.inf,
        }
        best_time = min(times.values())
        if best_time == np.inf:
            continue
        best_impl = min(times, key=times.get)

        x = np.array([[num_frames, width, height, total_pixels,
                        pixels_per_frame, rank, thread, node, int(cuda)]])
        pred_label = tree.predict(x)[0]
        pred_impl = imp[pred_label]
        pred_time = times[pred_impl]

        if pred_time == np.inf:
            continue

        regret = (pred_time - best_time) / best_time
        regret_records.append((regret, pred_impl, best_impl, pred_time, best_time,
                               gif_file, rank, thread, node, cuda))

regrets = np.array([r[0] for r in regret_records])
exact_best = (regrets == 0).sum()
total = len(regrets)
from scipy.stats import trim_mean

print(f"Samples: {total}, exact best: {exact_best} ({100*exact_best/total:.1f}%)")
print(f"\nRegret (predicted is X% slower than optimal):")
print(f"  Mean (trimmed 5%): {trim_mean(regrets, 0.05)*100:.1f}%")
print(f"  Median:            {np.median(regrets)*100:.1f}%")
print(f"  P90:               {np.percentile(regrets, 90)*100:.1f}%")
print(f"  P95:               {np.percentile(regrets, 95)*100:.1f}%")
print(f"  P99:               {np.percentile(regrets, 99)*100:.1f}%")
print(f"  Max:               {regrets.max()*100:.1f}%")
print(f"\n  Within 10%: {(regrets <= 0.10).mean()*100:.1f}% of samples")
print(f"  Within 20%: {(regrets <= 0.20).mean()*100:.1f}% of samples")
print(f"  Within 50%: {(regrets <= 0.50).mean()*100:.1f}% of samples")

# Top 10 worst cases
print("\nTop 10 worst regret cases:")
print(f"  {'Regret':>8}  {'Pred':>5} {'Best':>5}  {'Pred_t':>7} {'Best_t':>7}  {'GIF':<40}  R   T   N  CUDA")
sorted_records = sorted(regret_records, key=lambda r: -r[0])
for r in sorted_records[:10]:
    regret, pred, best, pt, bt, gif, rank, thread, node, cuda = r
    print(f"  {regret*100:7.1f}%  {pred:>5} {best:>5}  {pt:7.3f} {bt:7.3f}  {gif:<40}  {rank:<3} {thread:<3} {node:<3} {cuda}")

Samples: 2700, exact best: 2188 (81.0%)

Regret (predicted is X% slower than optimal):
  Mean (trimmed 5%): 5.3%
  Median:            0.0%
  P90:               34.8%
  P95:               110.1%
  P99:               1537.6%
  Max:               14327.5%

  Within 10%: 84.5% of samples
  Within 20%: 85.9% of samples
  Within 50%: 91.7% of samples

Top 10 worst regret cases:
    Regret   Pred  Best   Pred_t  Best_t  GIF                                       R   T   N  CUDA
  14327.5%    MPI   HYB    0.125   0.001  200_s.gif                                 4   1   2   True
  12696.5%    MPI   OMP    0.125   0.001  200_s.gif                                 4   2   2   True
   8885.7%    MPI   OMP    0.087   0.001  200_s.gif                                 4   2   4   True
   8062.0%    MPI   OMP    4.671   0.057  TimelyHugeGnu.gif                         2   2   4   False
   7958.2%    MPI   OMP    0.087   0.001  200_s.gif                                 4   1   4   True
   6740.9%    MPI  

In [ ]:
fig, ax = plt.subplots(figsize=(28, 14), dpi=150)
plot_tree(tree, feature_names=feature_names, class_names=class_names,
          filled=True, rounded=True, fontsize=8, ax=ax,
          proportion=True, impurity=False)
fig.tight_layout()
os.makedirs("visuals", exist_ok=True)
fig.savefig("visuals/decision_tree.png", bbox_inches="tight")
fig.savefig("visuals/decision_tree.pdf", bbox_inches="tight")
print("Saved visuals/decision_tree.png and visuals/decision_tree.pdf")
plt.show()